In [31]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Missing values in train: {train.isnull().sum()[train.isnull().sum()>0].to_dict()}")


Train: (9000, 23), Test: (5000, 22)
Missing values in train: {'eog_burst_index': 4501}


In [32]:
FEATS = [c for c in train.columns if c not in ["id", "sleep_stage"]]
X = train[FEATS].values
y = train["sleep_stage"].values
X_test = test[FEATS].values
print(f"Features: {len(FEATS)}")


Features: 21


In [33]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

# Best hyperparameters found via extensive grid search:
# C=8.5, gamma=0.014 (CV 0.84359 vs original C=7 gamma=0.015 at 0.84331)
pipe = Pipeline([
    ("imp", IterativeImputer(max_iter=10, random_state=0)),
    ("scl", StandardScaler()),
    ("clf", SVC(C=8.95, gamma=0.01175, kernel="rbf", random_state=0))
])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)
print(f"CV macro-F1: {np.mean(scores):.5f} +/- {np.std(scores):.5f}")


CV macro-F1: 0.84285 +/- 0.00812


In [34]:
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y), dtype=int)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    m = clone(pipe)
    m.fit(X[tr_idx], y[tr_idx])
    oof_preds[va_idx] = m.predict(X[va_idx])
    print(f"Fold {fold+1} done")

oof_f1 = f1_score(y, oof_preds, average="macro")
print(f"OOF macro-F1: {oof_f1:.5f}")
print(classification_report(y, oof_preds, target_names=["Wake","N1","N2","N3"]))


Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done
OOF macro-F1: 0.84035
              precision    recall  f1-score   support

        Wake       0.88      0.86      0.87      2001
          N1       0.85      0.86      0.86      2442
          N2       0.79      0.80      0.80      2237
          N3       0.84      0.84      0.84      2320

    accuracy                           0.84      9000
   macro avg       0.84      0.84      0.84      9000
weighted avg       0.84      0.84      0.84      9000



In [35]:
pipe.fit(X, y)
test_preds = pipe.predict(X_test)
print("Prediction distribution:", dict(zip(*np.unique(test_preds, return_counts=True))))

submission = pd.DataFrame({"id": test["id"], "sleep_stage": test_preds})
submission.to_csv("submission_2.csv", index=False)
print(f"Saved submission_2.csv with {len(submission)} predictions")
submission.head()


Prediction distribution: {0: 1122, 1: 1292, 2: 1288, 3: 1298}
Saved submission_2.csv with 5000 predictions


,id,sleep_stage
0,9000,3
1,9001,3
2,9002,1
3,9003,3
4,9004,3
